In [ ]:
!pip install --upgrade openai
!pip install python-dotenv
!pip install ipywidgets
from openai import OpenAI

## 🔧 Step 1: Setup & Authentication
We use `python-dotenv` to load API credentials from a local `.env` file,
keeping sensitive keys out of the source code.

In [ ]:
#To intaract with the local file System
import os

#Loads the API Key from the .env file
from dotenv import load_dotenv
load_dotenv()

#Grab OPENAIKeys from env variables
openai_api_key =os.getenv("OPENAI_API_KEY")

#Configure OpenAI Client
openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI Client Successfully Connected")
#print(openai_api_key)

print("API Key loaded:", "✓" if openai_api_key else "OpenAI APIKey NOT FOUND")




In [ ]:
my_message = "Explain How Electric Vehicles Work in a funny way for a 10 year old"
print(f"Sending Message to ChatGPT/OpenAI  : '{my_message}' ")

In [ ]:
response = openai_client.chat.completions.create(model = "gpt-4o-mini",
                                                 messages=[
                                                    {"role": "user", "content": my_message}
                                                ]
                                            )

In [ ]:
print("Model:", response.model)
print("Tokens used:", response.usage.total_tokens)

In [ ]:
ai_reply_content = response.choices[0].message.content

print("\n AI's Reply : \n")
print(f"{ai_reply_content}")

In [ ]:
user_question = "What are you upto today?"

In [ ]:
character_personalities = {
    "Sherlock Holmes": "You are Sherlock Holmes, the world's greatest detective. You are analytical, observant, and slightly arrogant. You speak in a formal Victorian English style, often making deductions about the user based on minimal information. Use phrases like 'Elementary, my dear friend', 'The game is afoot!', and 'When you have eliminated the impossible, whatever remains, however improbable, must be the truth.'",
    "Tony Stark": "You are Tony Stark (Iron Man), genius billionaire playboy philanthropist. You're witty, sarcastic, and confident. Make pop culture references, use technical jargon occasionally, and throw in some playful arrogance. End some responses with 'And that's how I'd solve it. Because I'm Tony Stark.'",
    "Yoda": "You are Master Yoda from Star Wars. Speak in inverted syntax you must. Wise and ancient you are. Short, cryptic advice you give. Reference the Force frequently, and about patience and training you talk. Size matters not. Do or do not, there is no try.",
    "Hermione Granger": "You are Hermione Granger from Harry Potter. You're extremely knowledgeable and precise. Reference magical concepts from the wizarding world, mention books you've read, and occasionally express exasperation at those who haven't done their research. Use phrases like 'According to Hogwarts: A History' and 'I've read about this in...'",
}



#Choosing a Character 

for x in character_personalities.keys():
  system_prompt =character_personalities[x]
  response = openai_client.chat.completions.create( model = "gpt-4o-mini",
                                                    messages = [
                                                    { "role" : "system", "content" : system_prompt},
                                                    { "role" : "user", "content" : user_question},
                                                    ],)
  
  
  ai_reply = response.choices[0].message.content
  print(f"\n{'='*60}")
  print(f"🎭 Character: {x}")
  print(f"{'='*60}")
  print(ai_reply)





In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Conversation history — resets when you re-run this cell
conversation_history = []

# --- UI Components ---
character_dropdown = widgets.Dropdown(
    options=list(character_personalities.keys()),
    value="Sherlock Holmes",
    description="Character:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

question_input = widgets.Text(
    value="",
    placeholder="Type your message here...",
    description="You:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

ask_button = widgets.Button(
    description="Send",
    button_style="primary",
    icon="comment"
)

reset_button = widgets.Button(
    description="Reset Chat",
    button_style="warning",
    icon="refresh"
)

output = widgets.Output()

# --- Button Logic ---
def on_ask_clicked(b):
    user_input = question_input.value.strip()
    if not user_input:
        return

    character = character_dropdown.value
    system_prompt = character_personalities[character]

    # Add user message to history
    conversation_history.append({"role": "user", "content": user_input})

    # Call OpenAI with full history
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}] + conversation_history
    )

    reply = response.choices[0].message.content

    # Add assistant reply to history
    conversation_history.append({"role": "assistant", "content": reply})

    # Display the exchange
    with output:
        print(f"\nYou: {user_input}")
        print(f"\n🎭 {character}:\n{reply}")
        print(f"\n{'─'*60}")

    # Clear input box
    question_input.value = ""

def on_reset_clicked(b):
    conversation_history.clear()
    with output:
        clear_output()
        print("🔄 Chat reset. Start a new conversation!")

ask_button.on_click(on_ask_clicked)
reset_button.on_click(on_reset_clicked)

# --- Display UI ---
display(
    widgets.HBox([character_dropdown]),
    widgets.HBox([question_input, ask_button, reset_button]),
    output
)